# Session 2: how stiff, how strong, how light?
**A. Real-world problem.** Two planks of the same wood and the same width span a stream.
One lies flat, the other stands on edge. The one on edge hardly bends. Why, when both
contain the same amount of wood? And if a building code says the beam may sag at most
L/360 and the wood may not be stressed beyond a limit, what is the lightest beam that passes?

**Plan.** (1) second moment of area I of a rectangle and an I-section, (2) deflection of a
point-loaded beam, (3) bending stress, (4) search for the lightest section that passes.
Units: N, m, Pa inside the functions; the printouts convert to mm and MPa.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"
RHO = 7850.0     # steel density, kg/m^3 (used for mass per metre)

## Step 1: the second moment of area I
Material far from the middle line (the *neutral axis*) resists bending much more than material
near it, because each small piece of area dA counts as y^2 dA.

    rectangle b wide, h deep:   I = b h^3 / 12
    I-section (flanges bf x tf, web tw, depth h):   I = [ bf h^3 - (bf - tw)(h - 2 tf)^3 ] / 12

Depth enters as h^3: doubling the depth multiplies I by 8, doubling the width only by 2.

In [ ]:
def I_rect(b, h):
    """Second moment of area of a solid rectangle, m^4 (b, h in m)."""
    return b * h ** 3 / 12.0


def I_isection(bf, tf, tw, h):
    """Second moment of area of a symmetric I-section, m^4 (all lengths in m)."""
    return (bf * h ** 3 - (bf - tw) * (h - 2 * tf) ** 3) / 12.0


def area_isection(bf, tf, tw, h):
    return 2 * bf * tf + tw * (h - 2 * tf)

## Step 2: deflection of a point-loaded, simply supported beam
With load P at distance a from the left support (b = L - a), E the modulus (Pa) and I (m^4),
the curve (Euler-Bernoulli beam theory, small deflections) is, for x <= a:

    v(x) = P b x (L^2 - b^2 - x^2) / (6 E I L)

and for x > a the same formula with x measured from the right support and a and b swapped.
For a centred load the maximum is  v_max = P L^3 / (48 E I)  at mid-span.

In [ ]:
def deflection(x, P, a, L, E, I):
    """Downward deflection v(x) in m of a simply supported beam with a point load."""
    x = np.asarray(x, dtype=float)
    b = L - a
    left = P * b * x * (L ** 2 - b ** 2 - x ** 2) / (6 * E * I * L)
    xr = L - x
    right = P * a * xr * (L ** 2 - a ** 2 - xr ** 2) / (6 * E * I * L)
    return np.where(x <= a, left, right)


def max_deflection(P, a, L, E, I, n=2001):
    x = np.linspace(0, L, n)
    return float(deflection(x, P, a, L, E, I).max())


def centre_deflection(P, L, E, I):
    return P * L ** 3 / (48 * E * I)

## Step 3: bending stress
The bending moment M makes the top fibres squash and the bottom fibres stretch. At a distance
y from the neutral axis the stress is sigma = M y / I. The largest value is at y = h / 2:

    sigma_max = M_max (h/2) / I,      M_max = P a b / L.

In [ ]:
def moment_max(P, a, L):
    return P * a * (L - a) / L


def bending_stress(M, y, I):
    """Stress in Pa at distance y (m) from the neutral axis."""
    return M * y / I

## Step 4: the design problem
Pass two tests: deflection <= L/360 and stress <= allowable. Among passing rectangles of
fixed width, the lightest is the shallowest one that passes (mass per metre = rho * area).

In [ ]:
def check_rect(P, a, L, E, b, h, sigma_allow, ratio=360.0):
    I = I_rect(b, h)
    d = max_deflection(P, a, L, E, I)
    s = bending_stress(moment_max(P, a, L), h / 2, I)
    return d, s, (d <= L / ratio and s <= sigma_allow)


def lightest_rect(P, a, L, E, b, sigma_allow, ratio=360.0):
    """Smallest depth h (m, steps of 1 mm) of a rectangle of width b that passes both tests."""
    for mm in range(10, 1500):
        h = mm / 1000.0
        if check_rect(P, a, L, E, b, h, sigma_allow, ratio)[2]:
            return h
    return None

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    P, L, E = 10e3, 4.0, 10e9             # N, m, Pa (timber-like)
    b, h = 0.100, 0.200
    I = I_rect(b, h)
    d = centre_deflection(P, L, E, I)
    s = bending_stress(P * L / 4, h / 2, I)
    print(f"I = {I * 1e12:.4g} mm^4, centre deflection {d * 1e3:.2f} mm, limit L/360 = {L / 360 * 1e3:.2f} mm, stress {s / 1e6:.2f} MPa")
    print("flat vs on edge, 100 x 200 mm: I ratio =", round(I_rect(0.1, 0.2) / I_rect(0.2, 0.1), 2))
    print("+20% area as depth: I x", round(I_rect(0.1, 0.24) / I, 3), " as width: I x", round(I_rect(0.12, 0.2) / I, 3))
    hh = lightest_rect(P, L / 2, L, E, b, 12e6)
    print(f"lightest 100 mm wide rectangle: h = {hh * 1e3:.0f} mm", check_rect(P, L / 2, L, E, b, hh, 12e6))

    x = np.linspace(0, L, 401)
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    for a, c in ((1.0, "tab:orange"), (2.0, "tab:blue")):
        ax[0].plot(x, -1e3 * deflection(x, P, a, L, E, I), c, label=f"load at {a:g} m")
    ax[0].axhline(-1e3 * L / 360, color="r", ls="--", label="limit L/360")
    ax[0].set_xlabel("position along the beam x (m)")
    ax[0].set_ylabel("deflection (mm, downward is negative)")
    ax[0].set_title("Deflected shape, 100 x 200 mm timber")
    ax[0].grid(alpha=0.3)
    ax[0].legend(fontsize=8)
    hs = np.arange(0.12, 0.40, 0.005)
    dd = [1e3 * centre_deflection(P, L, E, I_rect(b, q)) for q in hs]
    ss = [bending_stress(P * L / 4, q / 2, I_rect(b, q)) / 1e6 for q in hs]
    ax[1].plot(hs * 1e3, dd, label="deflection (mm)")
    ax[1].plot(hs * 1e3, ss, label="max stress (MPa)")
    ax[1].axhline(L / 360 * 1e3, color="r", ls="--", label="deflection limit 11.1 mm")
    ax[1].axhline(12, color="g", ls=":", label="stress limit 12 MPa")
    ax[1].axvline(hh * 1e3, color="k", lw=0.8)
    ax[1].set_xlabel("beam depth h (mm), width 100 mm")
    ax[1].set_ylabel("deflection (mm) or stress (MPa)")
    ax[1].set_title("Deeper beams are much stiffer")
    ax[1].set_ylim(0, 30)
    ax[1].grid(alpha=0.3)
    ax[1].legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(OUT / "deflection_design.png", dpi=120)

    # I-section vs rectangle of the same area
    bf, tf, tw, H = 0.100, 0.010, 0.006, 0.200
    A = area_isection(bf, tf, tw, H)
    Ii = I_isection(bf, tf, tw, H)
    wr = A / H                      # solid rectangle with the same area AND the same depth
    Ir = I_rect(wr, H)
    print(f"I-section area {A * 1e6:.0f} mm^2, I {Ii * 1e12:.4g} mm^4; solid rectangle of same area and depth (width {wr * 1e3:.1f} mm) I {Ir * 1e12:.4g} mm^4; ratio {Ii / Ir:.2f}")
    fig2, ax2 = plt.subplots(figsize=(6, 4))
    ax2.bar(["solid rectangle", "I-section"], [Ir * 1e12 / 1e6, Ii * 1e12 / 1e6], color=["tab:gray", "tab:blue"])
    ax2.set_ylabel("second moment of area I (10^6 mm^4)")
    ax2.set_title(f"Same area and depth: material far from the axis wins")
    ax2.grid(alpha=0.3, axis="y")
    fig2.tight_layout()
    fig2.savefig(OUT / "section_shapes.png", dpi=120)

## Where this stops
We used one load and a formula. Real beams carry distributed loads, change section along
their length and meet many load cases. Session 3 solves those numerically and checks the
solver against the formulas you just used.

## Coding tasks
1. **Easy.** Use `lightest_rect` in `lesson.py` with E = 12 GPa and a width of 60 mm. Predict whether the answer is shallower or deeper than 244 mm first.
2. **Stretch.** Write a function that finds the lightest I-section among a list of depths for the same 10 kN, 4 m, E = 10 GPa case (use `I_isection` and `area_isection`).
3. **Explain.** Explain, with the formula I = b h^3 / 12, why a plank is stiffer on edge than flat, and why a floor joist is deep and narrow.

## Assessment
- **A. Conceptual.** Why does increasing a beam's depth often improve bending stiffness more effectively than adding the same amount of material near the neutral axis?
- **B. Numerical.** A timber beam 50 mm wide and 150 mm deep, E = 12 GPa, spans 3 m with a 5 kN load at mid-span. Find the maximum deflection and stress, with units, and compare with L/360 and 12 MPa.
- **C. Graph and simulation interpretation.** On the demand graph (demand divided by limit against depth) the blue curve crosses 1 at about 243 mm and the green curve at about 224 mm. Which limit governs and what does it mean for the designer?
- **D. Design and optimisation challenge.** Design: with E = 10 GPa, L = 4 m and the 10 kN centre load, compare two ways of reaching the required stiffness: keep the width at 100 mm and increase the depth, or keep the depth at 200 mm and increase the width. Which uses less material, and by how much?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    I = L.I_rect(0.100, 0.200)
    checks.append((f"worked example: I = b h^3 / 12 = 6.667e-5 m^4 (got {I:.4g})", abs(I - 6.6667e-5) < 1e-8))
    d = L.centre_deflection(10e3, 4.0, 10e9, I)
    checks.append((f"worked example: centre deflection P L^3 / (48 E I) = 20.0 mm (got {d * 1e3:.2f})", abs(d - 0.020) < 1e-6))
    s = L.bending_stress(L.moment_max(10e3, 2.0, 4.0), 0.100, I)
    checks.append((f"worked example: max stress M y / I = 15.0 MPa (got {s / 1e6:.2f})", abs(s - 15e6) < 1e3))
    checks.append(("general formula agrees with P L^3/(48 E I) for a centred load",
                   abs(L.max_deflection(10e3, 2.0, 4.0, 10e9, I) - d) < 1e-9))
    x = np.array([0.0, 4.0])
    checks.append(("deflection is zero at both supports", bool(np.all(np.abs(L.deflection(x, 10e3, 1.3, 4.0, 10e9, I)) < 1e-12))))
    checks.append(("doubling depth multiplies I by 8", abs(L.I_rect(0.1, 0.4) / L.I_rect(0.1, 0.2) - 8) < 1e-12))
    checks.append(("doubling width multiplies I by 2", abs(L.I_rect(0.2, 0.2) / L.I_rect(0.1, 0.2) - 2) < 1e-12))
    checks.append(("+20% area as depth gives more I than +20% as width", L.I_rect(0.1, 0.24) > L.I_rect(0.12, 0.2)))
    checks.append(("deflection is inversely proportional to E",
                   abs(L.centre_deflection(10e3, 4.0, 20e9, I) * 2 - d) < 1e-12))
    # I-section: with tf = h/2 and tw = bf it degenerates to a solid rectangle
    checks.append(("I-section formula reduces to b h^3 / 12 when the web fills the section",
                   abs(L.I_isection(0.1, 0.1, 0.1, 0.2) - L.I_rect(0.1, 0.2)) < 1e-12))
    Ii = L.I_isection(0.100, 0.010, 0.006, 0.200)
    Ir = L.I_rect(L.area_isection(0.100, 0.010, 0.006, 0.200) / 0.200, 0.200)
    checks.append(("I-section beats a solid rectangle of equal area and depth", Ii > 1.9 * Ir))
    h = L.lightest_rect(10e3, 2.0, 4.0, 10e9, 0.100, 12e6)
    checks.append((f"design: lightest 100 mm wide rectangle is 244 mm deep (got {h * 1e3:.0f})", abs(h - 0.244) < 1e-9))
    d1, s1, ok1 = L.check_rect(10e3, 2.0, 4.0, 10e9, 0.100, 0.243, 12e6)
    checks.append(("design: 243 mm fails the deflection limit", not ok1 and d1 > 4.0 / 360))
    checks.append(("edge case: a very shallow beam fails", not L.check_rect(10e3, 2.0, 4.0, 10e9, 0.100, 0.05, 12e6)[2]))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
